In [1]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import GWFish.modules as gw
import gc
import torch
from dingo.core.posterior_models.build_model import build_model_from_kwargs
import pandas as pd
from dingo.gw.inference.sampler import GWComposedSampler
import dingo.gw.injection as injection
from dingo.gw.noise.asd_dataset import ASDDataset
import shutil



os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # use GPU 1
os.environ["taskset -c"] = "0-48"  # use CPU cores 0-48
waveform_model = 'IMRPhenomXPHM'
f_ref = 20.
detectors = ['ET']
network = gw.detection.Network(detector_ids = detectors, detection_SNR = (0., 8.))


/home/tpausch/.conda/envs/ET-DINGO/lib/python3.10/site-packages/lalsimulation/_lalsimulation_swig.py:8: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


## Define functions

In [2]:
def simulate_injections(name, num_injections):

    pm = build_model_from_kwargs(
        filename=f"/scratch/tpausch/models/{name}/model_latest.pt",
        device="cuda",
        load_training_info=False,
    )

    
    out_dir = f"/scratch/tpausch/test_data/{name}/injections"
    os.makedirs(out_dir, exist_ok=True)

    np.random.seed(42)  # optional, for reproducibility

    injection_generator = injection.Injection.from_posterior_model_metadata(
        pm.metadata
    )

    asd_fname = "/scratch/tpausch/ASD/asds_ET_T8.hdf5"
    asd_dataset = ASDDataset(file_name=asd_fname)

    injection_generator.asd = {
        k: v[0] for k, v in asd_dataset.asds.items()
    }

    # This is the combined intrinsic + extrinsic bilby PriorDict
    # that the network was trained on
    prior = injection_generator.prior
    print(prior)

    # ============================================================
    # Draw and run injections
    # ============================================================

    for idx in range(num_injections):

        print(f"Running injection {idx}")

        theta = prior.sample()
        theta = {k: float(v) for k, v in theta.items()}  # cast numpy -> float

        strain_data = injection_generator.injection(theta)

        payload = {
            "theta_true": theta,
            "strain_data": strain_data,
        }

        fname = os.path.join(out_dir, f"inj_{idx:04d}.pkl")

        with open(fname, "wb") as f:
            pickle.dump(payload, f)

    print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
    print(torch.cuda.memory_reserved() / 1e9, "GB reserved")
    del pm
    gc.collect()
    torch.cuda.empty_cache()
    return 1 

In [ ]:
#TODO create template libray

In [ ]:
#TODO apply template matching to injections to give approxmate for chirp mass
#maybe use pcbc for that like

# from pycbc.waveform import get_fd_waveform
# from pycbc.filter import matched_filter

# # Calculate frequency resolution
# delta_f = 1.0 / conditioned.duration

# # Generate frequency-domain waveform
# hp, hc = get_fd_waveform(
#     approximant='IMRPhenomD',
#     mass1=25,
#     mass2=20,
#     delta_f=delta_f,           # Frequency resolution (must match data)
#     f_lower=20                 # Lower frequency cutoff (Hz)
# )

# # Resize template to match PSD length
# hp.resize(len(psd))

# # Perform matched filtering
# snr = matched_filter(
#     hp,
#     conditioned,
#     psd=psd,
#     low_frequency_cutoff=20
# )

# # Find peak SNR
# import numpy as np
# peak_idx = np.argmax(abs(snr).numpy())
# peak_snr = abs(snr[peak_idx])